In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import statsmodels.api as sm

df = pd.read_csv("data/migros_for_analysis.csv")

print("shape:", df.shape)
df.info()


shape: (3187, 11)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3187 entries, 0 to 3186
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   postal_code         3187 non-null   int64  
 1   population          3187 non-null   float64
 2   Aldi                3187 non-null   float64
 3   Coop                3187 non-null   float64
 4   Denner              3187 non-null   float64
 5   Lidl                3187 non-null   float64
 6   Migros              3187 non-null   float64
 7   total_stores        3187 non-null   float64
 8   weighted_taxpayers  3187 non-null   float64
 9   weighted_income     3187 non-null   float64
 10  competitor_count    3187 non-null   float64
dtypes: float64(10), int64(1)
memory usage: 274.0 KB


In [2]:
pd.options

In [3]:
df.head()

,postal_code,population,Aldi,Coop,Denner,Lidl,Migros,total_stores,weighted_taxpayers,weighted_income,competitor_count
0,1000,4248.0,0.0,0.0,0.0,0.0,0.0,0.0,2236.0,1.278653e+08,0.0
1,1003,6879.0,2.0,1.0,1.0,2.0,2.0,8.0,3621.0,2.070586e+08,6.0
2,1004,31463.0,0.0,2.0,1.0,0.0,3.0,6.0,16560.0,9.470397e+08,3.0
3,1005,12454.0,0.0,3.0,0.0,0.0,0.0,3.0,6555.0,3.748667e+08,3.0
4,1006,15621.0,0.0,2.0,2.0,0.0,1.0,5.0,8222.0,4.701938e+08,4.0


In [4]:
int_columns = ["Aldi", "Coop", "Denner", "Lidl", "Migros", "total_stores", "population"]

df[int_columns] = df[int_columns].astype(int)
df[int_columns].dtypes


Aldi            int64
Coop            int64
Denner          int64
Lidl            int64
Migros          int64
total_stores    int64
population      int64
dtype: object

In [5]:
# #group brand (Migros Group vs external competitor) 
df["migros_group_count"] = df["Migros"] + df["Denner"]          # Denner is in Migros Group
df["external_competitor_count"] = df["Aldi"] + df["Coop"] + df["Lidl"]

# indicator "underserved area"
df["has_migros"] = df["Migros"] > 0
df["population_per_migros"] = df["population"] / df["Migros"].replace(0, np.nan)
df["population_per_migros"] = df["population_per_migros"].fillna(df["population"])  # ไม่มี Migros = ประชากรทั้งหมดคือ "ตลาดที่ยังไม่มีใครจับ"

# average income
df["avg_income_per_taxpayer"] = df["weighted_income"] / df["weighted_taxpayers"].replace(0, np.nan)

df[["postal_code", "population", "Migros", "migros_group_count", "external_competitor_count",
    "has_migros", "population_per_migros", "avg_income_per_taxpayer"]].head(10)


,postal_code,population,Migros,migros_group_count,external_competitor_count,has_migros,population_per_migros,avg_income_per_taxpayer
0,1000,4248,0,0,0,False,4248.000000,57184.820203
1,1003,6879,2,3,5,True,3439.500000,57182.723452
2,1004,31463,3,4,2,True,10487.666667,57188.387198
3,1005,12454,0,0,3,False,12454.000000,57187.908872
4,1006,15621,1,3,2,True,15621.000000,57187.276099
5,1007,22716,4,5,2,True,5679.000000,57189.197808
6,1008,14475,2,3,6,True,7237.500000,64896.608528
7,1009,19396,1,1,1,True,19396.000000,86329.952767
8,1010,15745,1,3,2,True,15745.000000,57189.115175
9,1011,55,0,0,0,False,55.000000,57086.411362


In [6]:
print(df["avg_income_per_taxpayer"].describe())
print()
print("postcodes with inf or weird avg_income:", (~np.isfinite(df["avg_income_per_taxpayer"])).sum())


count      3145.000000
mean      67146.602283
std       13678.459059
min       33838.933333
25%       58534.624842
50%       65046.095171
75%       72039.259211
max      234227.167513
Name: avg_income_per_taxpayer, dtype: float64

postcodes with inf or weird avg_income: 42


## Act 1 — Where does Migros stand today?


In [7]:
total_postcodes = len(df)
total_population = df["population"].sum()
total_stores = df[["Aldi", "Coop", "Denner", "Lidl", "Migros"]].sum().sum()
migros_stores = df["Migros"].sum()
postcodes_with_migros = (df["Migros"] > 0).sum()
population_with_migros = df.loc[df["Migros"] > 0, "population"].sum()

print(f"Total postal codes in Switzerland: {total_postcodes:,}")
print(f"Total population represented: {total_population:,.0f}")
print(f"Total supermarkets (5 brands): {total_stores:,.0f}")
print(f"  of which Migros: {migros_stores:,.0f} ({migros_stores/total_stores*100:.1f}%)")
print(f"Postal codes with a Migros: {postcodes_with_migros:,} ({postcodes_with_migros/total_postcodes*100:.1f}%)")
print(f"Population living in a Migros-covered postcode: {population_with_migros:,.0f} ({population_with_migros/total_population*100:.1f}%)")


Total postal codes in Switzerland: 3,187
Total population represented: 9,127,125
Total supermarkets (5 brands): 2,795
  of which Migros: 717 (25.7%)
Postal codes with a Migros: 550 (17.3%)
Population living in a Migros-covered postcode: 5,456,349 (59.8%)


**Key insight:** Migros already reaches 59.8% of the Swiss population despite having
stores in only 17.3% of postal codes — stores are concentrated where people already
are. The remaining **40.2% of the population (~3.67M people)** live in postal codes
with zero Migros presence.


In [8]:
brand_totals = df[["Aldi", "Coop", "Denner", "Lidl", "Migros"]].sum().reset_index()
brand_totals.columns = ["brand", "store_count"]

color_map = {"Migros": "#E69F00", "Coop": "#B0B0B0", "Denner": "#B0B0B0", "Aldi": "#B0B0B0", "Lidl": "#B0B0B0"}

fig = px.bar(
    brand_totals.sort_values("store_count", ascending=False),
    x="brand", y="store_count",
    color="brand", color_discrete_map=color_map,
    title="Supermarket count by brand — Switzerland",
)
fig.update_layout(showlegend=False)
fig.show()


In [9]:
coverage_compare = pd.DataFrame({
    "metric": ["% of postal codes", "% of population"],
    "coverage_pct": [
        postcodes_with_migros / total_postcodes * 100,
        population_with_migros / total_population * 100,
    ],
})

fig = px.bar(
    coverage_compare, x="metric", y="coverage_pct",
    color_discrete_sequence=["#E69F00"],
    title="Migros reaches far more people than postal codes it covers",
    text="coverage_pct",
)
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.update_layout(yaxis_title="Coverage (%)", xaxis_title="")
fig.show()


In [10]:
# compare group Migros and non-group 
gap_brand_only = ((df["Migros"] == 0) & (df["external_competitor_count"] + df["Coop"] + df["Aldi"] + df["Lidl"] > 0)).sum()
gap_migros_group = ((df["migros_group_count"] == 0) & (df["external_competitor_count"] > 0)).sum()

compare = pd.DataFrame({
    "definition": ["Migros brand only", "Migros Group (incl. Denner)"],
    "gap_postcode_count": [gap_brand_only, gap_migros_group],
})

compare


,definition,gap_postcode_count
0,Migros brand only,312
1,Migros Group (incl. Denner),224


**Difference: 88 postcodes** already have a Denner but no Migros-branded store.
Whether that counts as a "gap" depends on the question — a new Migros store may
still make sense there (different format/price point). Both `Migros` and
`migros_group_count` are kept so either framing can be used later.


## Act 2 — Where are customers and purchasing power?

In [11]:
fig = px.histogram(
    df, x="avg_income_per_taxpayer", nbins=50,
    color_discrete_sequence=["#0072B2"],
    title="Distribution of average income per taxpayer, per postcode",
)
fig.update_layout(xaxis_title="Average income per taxpayer (CHF)", yaxis_title="Number of postal codes")
fig.show()


In [12]:
fig = px.scatter(
    df, x="population", y="avg_income_per_taxpayer",
    size="total_stores",
    color_discrete_sequence=["#0072B2"],
    hover_data=["postal_code"],
    title="Population vs. average income per taxpayer, per postcode",
)
fig.update_layout(xaxis_title="Population", yaxis_title="Average income per taxpayer (CHF)")
fig.show()


In [13]:
print("correlation population vs avg_income_per_taxpayer:", df["population"].corr(df["avg_income_per_taxpayer"]))


correlation population vs avg_income_per_taxpayer: 0.10825008947664591


**Key insight:** Population size and income per capita are essentially independent
(r = 0.11) — a postcode having many people doesn't predict whether they're wealthy
or not, and vice versa. This means "customer volume" and "customer wealth" are two
separate dimensions to weigh when picking a location, not one that implies the other.


## Act 3 — Is Migros already where the demand is?


In [14]:
fig = px.scatter(
    df, x="population", y="Migros",
    size="weighted_income",
    color_discrete_sequence=["#E69F00"],
    hover_data=["postal_code", "competitor_count"],
    title="Population vs. Migros store count per postcode",
)
fig.update_layout(xaxis_title="Population", yaxis_title="Number of Migros stores")
fig.show()


In [15]:
print("correlation population vs Migros count:", df["population"].corr(df["Migros"]))
print("correlation competitor_count vs Migros count:", df["competitor_count"].corr(df["Migros"]))


correlation population vs Migros count: 0.7850464211238783
correlation competitor_count vs Migros count: 0.7625311851432802


**Key insight:** Migros presence correlates strongly with both population
(r = 0.785) and competitor presence (r = 0.763) — it already follows a
"go where the crowd is" logic. So sorting by population or competition alone
mostly returns places Migros is already in. The real gaps are **exceptions** —
high demand, lower-than-expected Migros presence. Act 4 uses regression to
isolate them.


## Act 4 — Where are the real gaps?

Use OLS regression to predict "expected" Migros store count from population,
income, and competitor presence. Postcodes where the **actual** count is much
lower than **predicted** are the real candidates — not just any high-population area,
but ones where Migros is under-represented *relative to what the pattern predicts*.


In [16]:
X = df[["population", "avg_income_per_taxpayer", "competitor_count"]].copy()
#้data have NaN value which cannot put in the model
X["avg_income_per_taxpayer"] = X["avg_income_per_taxpayer"].fillna(X["avg_income_per_taxpayer"].median())
X = sm.add_constant(X)
y = df["Migros"]

model = sm.OLS(y, X).fit()
print(model.summary())


                            OLS Regression Results                            
Dep. Variable:                 Migros   R-squared:                       0.656
Model:                            OLS   Adj. R-squared:                  0.656
Method:                 Least Squares   F-statistic:                     2025.
Date:                Fri, 18 Sep 2026   Prob (F-statistic):               0.00
Time:                        00:06:08   Log-Likelihood:                -1026.5
No. Observations:                3187   AIC:                             2061.
Df Residuals:                    3183   BIC:                             2085.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                              coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------
const                     

In [17]:
X = df[["population", "avg_income_per_taxpayer", "competitor_count"]].copy()
X["avg_income_per_taxpayer"] = X["avg_income_per_taxpayer"].fillna(X["avg_income_per_taxpayer"].median())
X = sm.add_constant(X)

df["predicted_migros"] = model.predict(X)
df["migros_residual"] = df["Migros"] - df["predicted_migros"]

top_gaps = df.sort_values("migros_residual").head(20)[
    ["postal_code", "population", "avg_income_per_taxpayer", "competitor_count",
     "Migros", "predicted_migros", "migros_residual"]
]
top_gaps


,postal_code,population,avg_income_per_taxpayer,competitor_count,Migros,predicted_migros,migros_residual
2538,8048,32652,77137.456021,14.0,1,3.828217,-2.828217
133,1205,35307,58949.880391,10.0,1,3.456763,-2.456763
140,1213,36322,58215.124800,9.0,1,3.378850,-2.378850
685,2300,37516,50928.784980,9.0,2,3.460620,-1.460620
152,1226,16940,62667.582916,3.0,0,1.391018,-1.391018
859,3008,11463,63636.495252,4.0,0,1.204619,-1.204619
2536,8046,26323,77135.760126,5.0,1,2.202117,-1.202117
1195,3800,16162,59168.303278,9.0,1,2.184814,-1.184814
2925,8854,11369,75297.727378,4.0,0,1.181234,-1.181234
1104,3600,17943,61779.519858,8.0,1,2.147101,-1.147101


In [18]:
fig = px.scatter(
    df, x="predicted_migros", y="Migros",
    color="migros_residual", color_continuous_scale="RdBu_r",
    hover_data=["postal_code"],
    title="Predicted vs. actual Migros count — color shows the gap",
)
fig.add_shape(type="line", x0=0, y0=0, x1=df["predicted_migros"].max(), y1=df["predicted_migros"].max(),
              line=dict(color="gray", dash="dash"))
fig.update_layout(xaxis_title="Predicted Migros count", yaxis_title="Actual Migros count")
fig.show()


In [19]:
fig = px.scatter(
    df, x="predicted_migros", y="migros_residual",
    color_discrete_sequence=["#0072B2"],
    title="Residuals vs. predicted — should scatter randomly around 0",
)
fig.add_hline(y=0, line_dash="dash", line_color="gray")
fig.show()


In [20]:
from sklearn.model_selection import train_test_split

train_idx, test_idx = train_test_split(df.index, test_size=0.2, random_state=42)

model_train = sm.OLS(y.loc[train_idx], X.loc[train_idx]).fit()
test_pred = model_train.predict(X.loc[test_idx])

r2_train = model_train.rsquared
r2_test = 1 - ((y.loc[test_idx] - test_pred)**2).sum() / ((y.loc[test_idx] - y.loc[test_idx].mean())**2).sum()

print(f"R² on training data: {r2_train:.3f}")
print(f"R² on unseen test data: {r2_test:.3f}")


R² on training data: 0.656
R² on unseen test data: 0.655


In [21]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif_data = pd.DataFrame({
    "variable": X.columns,
    "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
})
vif_data


,variable,VIF
0,const,25.656770
1,population,3.343302
2,avg_income_per_taxpayer,1.025765
3,competitor_count,3.306023


In [ ]:
import zipfile
from sklearn.neighbors import BallTree

# Load Migros for lat/lon
stores_df = pd.read_csv("data/switzerland_supermarkets_clean.csv")
migros_stores = stores_df[stores_df["store_group"] == "Migros"]

# create center of lat/lon postcode (nearest-neighbor imputation)
with zipfile.ZipFile("data/ortschaftenverzeichnis_plz_4326.csv.zip") as z:
    with z.open("AMTOVZ_CSV_WGS84/AMTOVZ_CSV_WGS84.csv") as f:
        raw_locations = pd.read_csv(f, sep=";", encoding="utf-8-sig")

postcode_coordinates_df = (
    raw_locations[["PLZ4", "E", "N"]]
    .rename(columns={"PLZ4": "postal_code", "E": "postcode_longitude", "N": "postcode_latitude"})
    .drop_duplicates(subset="postal_code", keep="first")
    .reset_index(drop=True)
)

# find distance from postcode to the nearest Migros
migros_rad = np.radians(migros_stores[["latitude", "longitude"]].to_numpy())
tree = BallTree(migros_rad, metric="haversine")

postcode_rad = np.radians(postcode_coordinates_df[["postcode_latitude", "postcode_longitude"]].to_numpy())
distance, index = tree.query(postcode_rad, k=1)
postcode_coordinates_df["distance_to_nearest_migros_km"] = distance.flatten() * 6371

df = df.merge(
    postcode_coordinates_df[["postal_code", "distance_to_nearest_migros_km"]],
    on="postal_code", how="left"
)

df["distance_to_nearest_migros_km"].describe()


count    3172.000000
mean        4.359882
std         4.973079
min         0.016930
25%         1.528253
50%         3.317494
75%         5.403144
max        48.249352
Name: distance_to_nearest_migros_km, dtype: float64

In [23]:
df[df["postal_code"].isin([8048, 8046, 8051, 8047])][
    ["postal_code", "Migros", "distance_to_nearest_migros_km"]
]


,postal_code,Migros,distance_to_nearest_migros_km
2536,8046,1,0.575238
2537,8047,1,0.335097
2538,8048,1,0.304487
2541,8051,1,0.511605


In [24]:
postcode_coordinates_df = (
    raw_locations[["PLZ4", "Ortschaftsname", "E", "N"]]
    .rename(columns={
        "PLZ4": "postal_code",
        "Ortschaftsname": "locality_name",
        "E": "postcode_longitude",
        "N": "postcode_latitude",
    })
    .drop_duplicates(subset="postal_code", keep="first")
    .reset_index(drop=True)
)


In [ ]:
migros_rad = np.radians(migros_stores[["latitude", "longitude"]].to_numpy())
tree = BallTree(migros_rad, metric="haversine")

postcode_rad = np.radians(postcode_coordinates_df[["postcode_latitude", "postcode_longitude"]].to_numpy())
distance, index = tree.query(postcode_rad, k=1)
postcode_coordinates_df["distance_to_nearest_migros_km"] = distance.flatten() * 6371

df = df.merge(
    postcode_coordinates_df[["postal_code", "locality_name"]], 
    on="postal_code", how="left"
)


In [26]:
df[["postal_code", "locality_name", "distance_to_nearest_migros_km"]].head(10)


,postal_code,locality_name,distance_to_nearest_migros_km
0,1000,Lausanne 25,1.491357
1,1003,Lausanne,0.163261
2,1004,Lausanne,0.210359
3,1005,Lausanne,0.983794
4,1006,Lausanne,0.238143
5,1007,Lausanne,0.402408
6,1008,Prilly,0.466345
7,1009,Pully,0.123814
8,1010,Lausanne,0.666020
9,1011,Lausanne,0.628111


In [27]:
top_gaps_full = df.sort_values("migros_residual").head(20)[
    ["postal_code", "locality_name", "population", "avg_income_per_taxpayer",
     "competitor_count", "Migros", "migros_residual", "distance_to_nearest_migros_km"]
]
top_gaps_full


,postal_code,locality_name,population,avg_income_per_taxpayer,competitor_count,Migros,migros_residual,distance_to_nearest_migros_km
2538,8048,Zürich,32652,77137.456021,14.0,1,-2.828217,0.304487
133,1205,Genève,35307,58949.880391,10.0,1,-2.456763,0.298848
140,1213,Petit-Lancy,36322,58215.124800,9.0,1,-2.378850,0.644809
685,2300,La Chaux-de-Fonds,37516,50928.784980,9.0,2,-1.460620,0.111754
152,1226,Thônex,16940,62667.582916,3.0,0,-1.391018,1.020538
859,3008,Bern,11463,63636.495252,4.0,0,-1.204619,1.101277
2536,8046,Zürich,26323,77135.760126,5.0,1,-1.202117,0.575238
1195,3800,Interlaken,16162,59168.303278,9.0,1,-1.184814,0.689469
2925,8854,Siebnen,11369,75297.727378,4.0,0,-1.181234,4.161277
1104,3600,Thun,17943,61779.519858,8.0,1,-1.147101,0.591837


In [28]:
DISTANCE_THRESHOLD_KM = 1.0

real_gaps = df[
    (df["migros_residual"] < 0) &
    (df["distance_to_nearest_migros_km"] > DISTANCE_THRESHOLD_KM)
].sort_values("migros_residual")

real_gaps_top20 = real_gaps.head(20)[
    ["postal_code", "locality_name", "population", "avg_income_per_taxpayer",
     "competitor_count", "Migros", "migros_residual", "distance_to_nearest_migros_km"]
]
real_gaps_top20


,postal_code,locality_name,population,avg_income_per_taxpayer,competitor_count,Migros,migros_residual,distance_to_nearest_migros_km
152,1226,Thônex,16940,62667.582916,3.0,0,-1.391018,1.020538
859,3008,Bern,11463,63636.495252,4.0,0,-1.204619,1.101277
2925,8854,Siebnen,11369,75297.727378,4.0,0,-1.181234,4.161277
3061,9244,Niederuzwil,6789,63439.329974,4.0,0,-0.928427,1.043537
1399,4415,Lausen,5996,66848.625751,4.0,0,-0.876305,2.089745
2658,8305,Dietlikon,8033,77319.861444,3.0,0,-0.841721,2.096506
1511,4632,Trimbach,6907,56137.760916,3.0,0,-0.807490,1.586067
39,1052,Le Mont-sur-Lausanne,9790,80137.317266,2.0,0,-0.802272,1.784196
1521,4663,Aarburg,9049,63016.404934,2.0,0,-0.784608,1.658035
2923,8852,Altendorf,7533,111522.047195,3.0,0,-0.759862,2.236567


In [29]:
df = df.merge(
    postcode_coordinates_df[["postal_code", "postcode_latitude", "postcode_longitude"]],
    on="postal_code", how="left"
)


In [ ]:
fig = px.scatter_mapbox(
    real_gaps_top20,
    lat="postcode_latitude", lon="postcode_longitude",
    size=real_gaps_top20["migros_residual"].abs(),
    color=real_gaps_top20["migros_residual"].abs(),
    color_continuous_scale="YlOrRd",   
    hover_name="locality_name",
    hover_data={"postal_code": True, "population": True, "distance_to_nearest_migros_km": ":.1f"},
    zoom=6.5, center={"lat": 46.8, "lon": 8.2},
    mapbox_style="open-street-map",
    title="Top 20 candidate locations for a new Migros store",
)
fig.update_layout(height=700, coloraxis_colorbar_title="Gap size")
fig.show()


C:\Users\ichay\AppData\Local\Temp\ipykernel_12952\4061482865.py:1: DeprecationWarning:

*scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/



In [31]:
real_gaps_top20[
    ["postal_code", "locality_name", "population", "avg_income_per_taxpayer",
     "competitor_count", "Migros", "migros_residual", "distance_to_nearest_migros_km"]
].reset_index(drop=True)


,postal_code,locality_name,population,avg_income_per_taxpayer,competitor_count,Migros,migros_residual,distance_to_nearest_migros_km
0,1226,Thônex,16940,62667.582916,3.0,0,-1.391018,1.020538
1,3008,Bern,11463,63636.495252,4.0,0,-1.204619,1.101277
2,8854,Siebnen,11369,75297.727378,4.0,0,-1.181234,4.161277
3,9244,Niederuzwil,6789,63439.329974,4.0,0,-0.928427,1.043537
4,4415,Lausen,5996,66848.625751,4.0,0,-0.876305,2.089745
5,8305,Dietlikon,8033,77319.861444,3.0,0,-0.841721,2.096506
6,4632,Trimbach,6907,56137.760916,3.0,0,-0.807490,1.586067
7,1052,Le Mont-sur-Lausanne,9790,80137.317266,2.0,0,-0.802272,1.784196
8,4663,Aarburg,9049,63016.404934,2.0,0,-0.784608,1.658035
9,8852,Altendorf,7533,111522.047195,3.0,0,-0.759862,2.236567


## Act 5 — Recommendation: where should Migros open next?


In [33]:
final_candidates = real_gaps.head(20)[
    ["postal_code", "locality_name", "population", "avg_income_per_taxpayer",
     "competitor_count", "Denner", "migros_residual", "distance_to_nearest_migros_km"]
].reset_index(drop=True)

final_candidates.index = final_candidates.index + 1   # rank เริ่มที่ 1 ไม่ใช่ 0
final_candidates


,postal_code,locality_name,population,avg_income_per_taxpayer,competitor_count,Denner,migros_residual,distance_to_nearest_migros_km
1,1226,Thônex,16940,62667.582916,3.0,2,-1.391018,1.020538
2,3008,Bern,11463,63636.495252,4.0,0,-1.204619,1.101277
3,8854,Siebnen,11369,75297.727378,4.0,1,-1.181234,4.161277
4,9244,Niederuzwil,6789,63439.329974,4.0,1,-0.928427,1.043537
5,4415,Lausen,5996,66848.625751,4.0,1,-0.876305,2.089745
6,8305,Dietlikon,8033,77319.861444,3.0,0,-0.841721,2.096506
7,4632,Trimbach,6907,56137.760916,3.0,1,-0.807490,1.586067
8,1052,Le Mont-sur-Lausanne,9790,80137.317266,2.0,0,-0.802272,1.784196
9,4663,Aarburg,9049,63016.404934,2.0,1,-0.784608,1.658035
10,8852,Altendorf,7533,111522.047195,3.0,1,-0.759862,2.236567


In [34]:
plot_df = final_candidates.copy()
plot_df["gap_size"] = plot_df["migros_residual"].abs()
plot_df["label"] = plot_df["locality_name"] + " (" + plot_df["postal_code"].astype(str) + ")"

fig = px.bar(
    plot_df.sort_values("gap_size"),
    x="gap_size", y="label",
    orientation="h",
    color_discrete_sequence=["#E69F00"],
    text="gap_size",
    hover_data={"population": True, "distance_to_nearest_migros_km": ":.1f"},
    title="Top 20 candidate locations, ranked by gap size",
)
fig.update_traces(texttemplate="%{text:.2f}", textposition="outside")
fig.update_layout(
    xaxis_title="Gap (expected − actual Migros stores)",
    yaxis_title="",
    height=700,
)
fig.show()


In [35]:
current_covered = df.loc[df["Migros"] > 0, "population"].sum()
added_pop = final_candidates["population"].sum()
total_pop = df["population"].sum()

impact = pd.DataFrame({
    "scenario": ["Today", "After 20 new stores"],
    "population_covered_pct": [
        current_covered / total_pop * 100,
        (current_covered + added_pop) / total_pop * 100,
    ],
})

fig = px.bar(
    impact, x="scenario", y="population_covered_pct",
    color_discrete_sequence=["#E69F00"],
    text="population_covered_pct",
    title="Impact: population living in a Migros-covered postcode",
)
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.update_layout(yaxis_title="Population covered (%)", xaxis_title="", yaxis_range=[0, 80])
fig.show()

print(f"Additional population reached: {added_pop:,.0f} people (+{added_pop/total_pop*100:.2f} percentage points)")


Additional population reached: 158,004 people (+1.73 percentage points)


In [36]:
sensitivity = []
for threshold in [0.5, 1.0, 2.0, 3.0]:
    subset = df[(df["migros_residual"] < 0) & (df["distance_to_nearest_migros_km"] > threshold)]
    top = subset.sort_values("migros_residual").head(20)
    sensitivity.append({
        "threshold_km": threshold,
        "qualifying_postcodes": len(subset),
        "top_candidate": top.iloc[0]["locality_name"],
        "median_population_top20": top["population"].median(),
    })

pd.DataFrame(sensitivity)


,threshold_km,qualifying_postcodes,top_candidate,median_population_top20
0,0.5,1914,Petit-Lancy,14308.0
1,1.0,1816,Thônex,7220.0
2,2.0,1491,Siebnen,5396.0
3,3.0,1108,Siebnen,4779.0


### Recommendation

Using a regression model (R² = 0.656, validated on held-out data) combined with a
geographic check, we identify **20 postal codes where Migros is under-represented
relative to local demand and where no Migros store exists within 1 km**.

- Top candidate: **Thônex (1226)** — 16,940 residents, no Migros, nearest store 1.0 km away
- Together these 20 locations would extend Migros coverage from **59.8% → 61.5%**
  of the Swiss population (+158,000 people)
- The geographic filter removed **17 of the 20** candidates the regression alone
  suggested — 15 of them already have a Migros inside the postcode itself
  (see the Zürich 8046–8051 check above), so the regression was flagging
  "fewer stores than expected", not "no store"


### Known limitations

1. **The 1 km threshold is a judgement call.** The sensitivity table shows the #1
   candidate changes (Petit-Lancy → Thônex → Siebnen) at 0.5 / 1.0 / 2.0 km.
2. **Rural postcodes are geographically larger**, so their centroid is naturally
   farther from any store.
3. **14 of the 20 candidates already have a Denner**, a Migros Group discount brand.
   They are kept because Denner serves a different price segment, but the Migros
   Group already has a physical presence there.
4. **No rent, land availability, or store-size data** — these are the decisive
   factors for an actual opening decision and must be checked on the ground.
5. Postcode-level analysis; store revenue per branch is not publicly available.
